# SageMaker AI — MLOps end-to-end (Experiments, HPO, Feature Store, Pipelines, Registry, Lineage)

**Course:** AWS DODEVA / MLGAIE
**Environment:** SageMaker AI Studio → JupyterLab → kernel **Python 3 (SageMaker Distribution)** (CPU)
**Estimated runtime:** ~55 min cell execution + ~35 min teaching = **90 min**
**Estimated cost:** a few US dollars at us-east-1 on-demand pricing.

---

## What you'll learn

Every MLOps surface SageMaker exposes, end-to-end, in one notebook:

1. **§3 Feature Store** — `FeatureGroup` with both **online** (low-latency lookup) and
   **offline** (S3 + Athena) stores. Sub-100ms reads on the hot path; ad-hoc analytics
   on the cold path.
2. **§4 Experiments** — three training runs tracked with the SageMaker Experiments SDK.
   Compare hyperparameters and metrics in the **Studio → Experiments** UI.
3. **§5 Hyperparameter Tuning** — `HyperparameterTuner` with **Bayesian** search.
   We discuss when Bayesian beats random and look at the trial-by-trial table.
4. **§6 Pipelines + ConditionStep** — `Preprocess → Train → Eval → ConditionStep` that
   branches on a metric threshold. Models above the bar register; models below the bar
   trigger a `FailStep` so the pipeline visibly fails in Studio.
5. **§7 Model Registry approval → auto-deploy** — flip `ModelApprovalStatus` to `Approved`,
   watch the **EventBridge rule** invoke a **Lambda** that creates a live endpoint, then
   hit it. This is the production-grade approval workflow most teams hand-roll badly.
6. **§8 Lineage Tracking** — query the artifact graph: endpoint → model package →
   training job → input artifacts. Cross-reference with the Studio Lineage UI.

> **Quota & cost note:** Processing/feature-engineering uses **`ml.t3.large`** (default
> quota: 4); training, experiment runs, tuning trials, and pipeline-training all use
> **`ml.m5.xlarge`** (gated by the global "4 training instances" quota); the auto-deployed
> endpoint uses **`ml.m5.large`** (default quota: 4). Tuning runs at most **2 trials in
> parallel** so we stay under the training quota. **Run §10 cleanup before logging off** —
> the live endpoint is the expensive bit.

> **Dataset:** California Housing (sklearn built-in). Regression target = median house value.

> **Prereq:** the `sagemaker-mlops-cfn.yaml` stack must already be deployed in this account.
> It provides the execution role, the S3 bucket, the auto-deploy Lambda, and — critically
> — the **EventBridge rule** that turns Model Registry approvals into live endpoints. If
> that rule is missing, §7 will look like the approval did nothing.

## §0. Diagnostic + resilient setup

We deliberately do NOT rely on `sagemaker.session.get_execution_role()` here — different
SageMaker Distribution images ship slightly different `sagemaker` layouts. boto3 + STS
gives us the role ARN unambiguously, then the SageMaker SDK is used only for the helpers
we actually need (Session, default bucket, Estimator, Pipeline, Tuner, FeatureGroup).

In [ ]:
# Import only stdlib + boto3 here. SageMaker SDK is probed defensively because some
# Distribution images ship 2.x and others ship a leftover 3.x preview that breaks
# half this notebook. We pin in §1 if needed.
import sys, os, importlib

print("Python:", sys.version.split()[0])

import boto3
print("boto3 :", boto3.__version__)

# 1. Probe the SageMaker SDK so we can see what's installed before any API call.
try:
    import sagemaker
    sm_version  = getattr(sagemaker, "__version__", "<no __version__>")
    sm_location = getattr(sagemaker, "__file__", "<no __file__>")
    print(f"sagemaker: {sm_version} @ {sm_location}")
except ImportError as e:
    print("sagemaker SDK not importable:", e)
    print("→ Run §1 install cell, RESTART KERNEL, then re-run §0.")
    raise

# SageMaker Distribution images sometimes ship `sagemaker` v3.x — a different,
# near-empty package on PyPI. It imports but has no Session/Estimator/etc.
# Detect it up front so the failure mode is one clear message, not a confusing
# AttributeError 50 lines down.
if not hasattr(sagemaker, "Session"):
    raise RuntimeError(
        "sagemaker package is missing Session — you have the v3.x stub installed.\n"
        "→ Run §1 below to install sagemaker>=2.246.0,<3.0, RESTART THE KERNEL, "
        "then re-run §0."
    )

# 2. Region — boto3 default first, then env var, then us-east-1 fallback.
region = boto3.session.Session().region_name or os.environ.get("AWS_REGION") or "us-east-1"
print(f"Region: {region}")

# 3. Account + role ARN via STS (avoids SDK layout quirks in get_execution_role).
sts = boto3.client("sts", region_name=region)
caller = sts.get_caller_identity()
account = caller["Account"]
arn     = caller["Arn"]

if os.environ.get("SAGEMAKER_ROLE"):
    # Local-execution override: pass an explicit SageMaker execution role ARN.
    # This is what verify_mlops.sh sets when running headless.
    role = os.environ["SAGEMAKER_ROLE"]
elif ":assumed-role/" in arn:
    # Inside Studio: the assumed-role ARN points to the execution role.
    role_name = arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
    role      = f"arn:aws:iam::{account}:role/{role_name}"
else:
    role = arn

print(f"Account: {account}")
print(f"Role   : {role}")

# 4. Default bucket — sagemaker.Session().default_bucket() picks
# sagemaker-{region}-{account} if the bucket exists or creates it.
# We *prefer* the MLOps demo bucket from the CFN stack if present.
sm_admin = boto3.client("sagemaker", region_name=region)
s3 = boto3.client("s3", region_name=region)

candidate = f"sagemaker-mlops-{account}-{region}"
try:
    s3.head_bucket(Bucket=candidate)
    bucket = candidate
    print(f"Bucket : {bucket} (CFN-provisioned)")
except Exception:
    try:
        session = sagemaker.Session()
        bucket  = session.default_bucket()
        print(f"Bucket : {bucket} (sagemaker default)")
    except Exception as e:
        bucket = f"sagemaker-{region}-{account}"
        print(f"Bucket : {bucket} (fallback — may not exist)")

# Standard sagemaker.Session for SDK calls that need it (Estimator, Tuner, etc.).
session = sagemaker.Session()

print("✓ setup OK")

## §1. Install / upgrade dependencies

SageMaker Distribution ships `sagemaker`, `boto3`, `xgboost`, `scikit-learn`. We pin a
recent SDK and add **pyathena** for the offline Feature Store query in §8. If the SDK
actually upgrades, **restart the kernel** before re-running §0.

In [ ]:
# Pin to sagemaker 2.x — version 3.x is a separate package with an incompatible API.
# pyathena: lightweight DB-API client over Athena (used in §8 to query the offline
# Feature Store table). It's not in the Distribution image by default.
%pip install --quiet --upgrade "sagemaker>=2.246.0,<3.0" "boto3>=1.34.0" "pyathena>=3.5"
print("✓ deps ready — if anything was upgraded, RESTART THE KERNEL before continuing")

## §2. Load California Housing → S3

sklearn has the dataset bundled — no S3 fetches, no Kaggle keys. Two formats are
produced:

- **For training** (§4, §5, §6): three CSVs (train/val/test) with target in column 0,
  no header — the format the built-in XGBoost container expects.
- **For Feature Store** (§3): a DataFrame with explicit `record_id` and `event_time`
  columns. Feature Store requires a unique identifier and a sortable event-time per row.

In [ ]:
import datetime as dt, os, io
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# RUN_ID drives every S3 prefix and resource name in this notebook so re-running
# the notebook never collides with a previous run's artifacts.
RUN_ID = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
PREFIX = "mlops-demo"
print(f"RUN_ID = {RUN_ID}")
print(f"PREFIX = {PREFIX}")

cal = fetch_california_housing(as_frame=True)
df  = cal.frame.copy()

# XGBoost CSV convention: target in column 0, no header.
target_col = "MedHouseVal"
feature_cols = [c for c in df.columns if c != target_col]

train_df, temp_df = train_test_split(df[[target_col] + feature_cols],
                                     test_size=0.25, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.40, random_state=42)
print(f"Rows train/val/test = {len(train_df)} / {len(val_df)} / {len(test_df)}")

work = "data"
os.makedirs(work, exist_ok=True)
raw_train = f"{work}/train.csv";      train_df.to_csv(raw_train, index=False, header=False)
raw_val   = f"{work}/validation.csv"; val_df.to_csv(raw_val,     index=False, header=False)
raw_test  = f"{work}/test.csv";       test_df.to_csv(raw_test,   index=False, header=False)

# Upload all three CSVs under a single run-scoped prefix.
raw_prefix = f"{PREFIX}/{RUN_ID}/raw"
for local in (raw_train, raw_val, raw_test):
    s3.upload_file(local, bucket, f"{raw_prefix}/{os.path.basename(local)}")

raw_uri       = f"s3://{bucket}/{raw_prefix}"
raw_train_uri = f"{raw_uri}/train.csv"
raw_val_uri   = f"{raw_uri}/validation.csv"
raw_test_uri  = f"{raw_uri}/test.csv"
print("Uploaded raw CSVs:")
for u in (raw_train_uri, raw_val_uri, raw_test_uri):
    print(" ", u)

In [ ]:
# Feature-Store-shaped DataFrame: needs a string record id + ISO-8601 event time.
# We'll use the entire training set (~15k rows) so §8's Athena query has substance.
fs_df = train_df.copy().reset_index(drop=True)
fs_df.insert(0, "record_id", [f"row-{i:06d}" for i in range(len(fs_df))])
# Feature Store needs a string ISO-8601 timestamp; one fixed value is fine for a
# snapshot demo. In a real system this would be the row's actual event time.
fs_df["event_time"] = dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"

print("Feature Store rows:", len(fs_df))
print(fs_df.head(3).to_string())

## §3. Feature Store — online + offline

A **FeatureGroup** is a typed table whose rows are **records** keyed by a `record_id`
and timestamped by an `event_time`. Two storage backends are configured side-by-side:

- **Online store** (DynamoDB under the hood): low-latency `get_record(record_id)` —
  sub-100ms — used by inference services that need a feature at request time.
- **Offline store** (S3 + Glue + Athena): every record streams to S3 in Parquet,
  partitioned by event time. Used for training, batch backfills, and analytics.

**Lifecycle quirk:** records ingested via `ingest()` arrive in the online store
immediately, but the offline store flushes asynchronously every ~5–15 minutes. We
ingest now and run the Athena query in §8, by which point the flush has happened.

In [ ]:
from sagemaker.feature_store.feature_group import FeatureGroup

fg_name = f"mlops-housing-{RUN_ID}"
print(f"FeatureGroup name: {fg_name}")

# Construct the FeatureGroup (no AWS call yet — that's .create()).
feature_group = FeatureGroup(name=fg_name, sagemaker_session=session)
# load_feature_definitions infers types from the DataFrame's dtypes.
# All numeric columns become Fractional, ints become Integral, strings become String.
feature_group.load_feature_definitions(data_frame=fs_df)

offline_s3_uri = f"s3://{bucket}/{PREFIX}/{RUN_ID}/feature-store"
print(f"Offline store URI: {offline_s3_uri}")

In [ ]:
# Create the FeatureGroup. Both online and offline are enabled.
# role_arn = the ExecutionRole — Feature Store assumes it to write to S3 + DDB.
feature_group.create(
    s3_uri=offline_s3_uri,
    record_identifier_name="record_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True,
    # disable_glue_table_creation=False is the default — we WANT the Glue
    # table so Athena can query the offline store in §8.
)

# Wait for the FeatureGroup to reach 'Created' status. ~5 min on first create.
import time
for i in range(60):
    status = feature_group.describe().get("FeatureGroupStatus")
    print(f"  status: {status} ({(i+1)*10}s elapsed)")
    if status == "Created":
        break
    if status == "CreateFailed":
        raise RuntimeError(feature_group.describe().get("FailureReason"))
    time.sleep(10)
print("✓ FeatureGroup ready")

In [ ]:
# Bulk-ingest the entire DataFrame. max_workers controls parallelism; the SDK
# batches PutRecord calls under the hood. Returns when all rows are accepted by
# the online store. The offline store flush happens asynchronously after.
ingestion = feature_group.ingest(data_frame=fs_df, max_workers=4, wait=True)
print(f"✓ ingested {len(fs_df)} records into online store")
if ingestion.failed_rows:
    print(f"⚠ {len(ingestion.failed_rows)} rows failed — check ingestion.failed_rows")

In [ ]:
# Online lookup demo — the hot path. Sub-100ms per call.
fs_runtime = boto3.client("sagemaker-featurestore-runtime", region_name=region)

sample_id = "row-000042"
resp = fs_runtime.get_record(FeatureGroupName=fg_name, RecordIdentifierValueAsString=sample_id)
print(f"Online lookup for {sample_id}:")
for f in resp.get("Record", []):
    print(f"  {f['FeatureName']:18s} = {f['ValueAsString']}")

## §4. Experiments — track three training runs

The SageMaker **Experiments** SDK (`sagemaker.experiments.Run`) wraps each training
job in a tracked Run that captures hyperparameters, metrics, and artifacts. Three
runs in one Experiment let us compare configurations side-by-side in
**Studio → Home → Experiments → mlops-housing-experiments**.

We run **3 short XGBoost training jobs** with different `max_depth` / `eta` settings.
Each one is a real training job (not local) so we exercise the whole pipeline:
container pull → fit → emit metrics → CloudWatch.

> Why three runs and not thirty? Same teaching point, 1/10th the cost.

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.image_uris import retrieve as image_retrieve
from sagemaker.experiments.run import Run

# XGBoost built-in container. Version 1.7-1 is current and supported in us-east-1.
xgb_image_uri = image_retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
)
print("XGBoost image URI:", xgb_image_uri)

EXPERIMENT_NAME = f"mlops-housing-{RUN_ID}"
print("Experiment   :", EXPERIMENT_NAME)

In [ ]:
# Three configs to compare. We pick deliberately distinct settings so the
# validation RMSEs spread visibly in the Experiments UI.
configs = [
    {"max_depth": 3, "eta": 0.10, "subsample": 0.8},
    {"max_depth": 5, "eta": 0.20, "subsample": 0.8},
    {"max_depth": 7, "eta": 0.30, "subsample": 0.9},
]

run_summaries = []

for i, hp in enumerate(configs, start=1):
    # SageMaker TrialComponent names must match [a-zA-Z0-9](-*[a-zA-Z0-9]){0,119}
    # (no dots) — encode eta as eta×100 integer so 0.1 → eta010, not e0.1.
    run_name = f"trial-{i:02d}-d{hp['max_depth']}-eta{int(hp['eta']*100):03d}"
    print(f"\n━━━━━━━━━━ {run_name} ━━━━━━━━━━")

    # Run() context manager links the training job to the experiment + run.
    # Every experiment-aware operation inside this `with` block is auto-tagged.
    with Run(experiment_name=EXPERIMENT_NAME,
             run_name=run_name,
             sagemaker_session=session) as run:

        run.log_parameters({"max_depth": hp["max_depth"],
                            "eta":       hp["eta"],
                            "subsample": hp["subsample"]})

        est = Estimator(
            image_uri=xgb_image_uri,
            role=role,
            instance_count=1,
            instance_type="ml.m5.xlarge",
            max_run=900,                  # 15 min cap per trial
            output_path=f"s3://{bucket}/{PREFIX}/{RUN_ID}/exp-{i}",
            base_job_name=f"exp-{i}",
            hyperparameters={
                "objective":   "reg:squarederror",
                "num_round":   60,         # short — we're sweeping configs, not maxing accuracy
                "max_depth":   hp["max_depth"],
                "eta":         hp["eta"],
                "subsample":   hp["subsample"],
                "eval_metric": "rmse",
            },
            # NOTE: built-in XGBoost auto-emits "validation:rmse" — passing
            # metric_definitions=[...] to a built-in algorithm Estimator raises
            # ValidationException ("You can't override the metric definitions").
            # We pull the metric back via TrainingJobAnalytics below.
        )

        est.fit({
            "train":      TrainingInput(raw_train_uri, content_type="text/csv"),
            "validation": TrainingInput(raw_val_uri,   content_type="text/csv"),
        }, wait=True, logs="None")  # logs="None" keeps the notebook output readable

        # Pull the final RMSE out of CloudWatch metrics and attach to the Run.
        from sagemaker.analytics import TrainingJobAnalytics
        metrics = TrainingJobAnalytics(training_job_name=est.latest_training_job.name).dataframe()
        final_rmse = float(metrics[metrics["metric_name"] == "validation:rmse"]["value"].iloc[-1])
        run.log_metric(name="validation:rmse", value=final_rmse)
        print(f"  → validation:rmse = {final_rmse:.4f}")

        run_summaries.append({"run": run_name, "rmse": final_rmse, **hp})

print("\n━━━━━━━━━━ Experiment summary ━━━━━━━━━━")
print(pd.DataFrame(run_summaries).to_string(index=False))

> **In Studio:** open the left **Home** menu → **Experiments**. Pick
> `mlops-housing-{RUN_ID}` and you'll see all three Runs side-by-side with their
> parameters and metrics. The chart view lets you scatter `max_depth` vs
> `validation:rmse` to see the bias/variance tradeoff visually.

## §5. Hyperparameter Tuning — Bayesian search

`HyperparameterTuner` runs many training jobs with hyperparameters drawn from ranges
you specify, optimizing for an objective metric. Two main strategies:

- **Random**: each trial is independent — embarrassingly parallel, but no learning.
- **Bayesian**: each trial is informed by previous trials' results — explores
  promising regions of parameter space more efficiently. Slower per-trial because
  Bayesian decisions are sequential.

**When to pick which?** Bayesian wins when (a) trials are expensive and (b) the
parameter landscape is somewhat smooth. Random wins when you have lots of compute,
cheap trials, or wide flat optima. We pick **Bayesian** here because each XGBoost
training job costs real money.

**Budget for the demo:** `max_jobs=4`, `max_parallel_jobs=2`. Bayesian needs at least
8–10 trials to really shine; we cap at 4 to fit the 90-minute window. The point is
the API + UI, not the optimization quality.

In [ ]:
from sagemaker.tuner import (
    HyperparameterTuner, ContinuousParameter, IntegerParameter,
)

tuner_estimator = Estimator(
    image_uri=xgb_image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    max_run=900,
    output_path=f"s3://{bucket}/{PREFIX}/{RUN_ID}/hpo",
    base_job_name="hpo",
    hyperparameters={
        "objective":   "reg:squarederror",
        "num_round":   100,
        "eval_metric": "rmse",
    },
    # No metric_definitions — built-in XGBoost auto-emits validation:rmse.
)

# Hyperparameter ranges. ContinuousParameter is uniform over [low, high];
# IntegerParameter is uniform over the integer interval.
hyperparameter_ranges = {
    "eta":       ContinuousParameter(0.05, 0.5),
    "max_depth": IntegerParameter(3, 8),
    "subsample": ContinuousParameter(0.5, 1.0),
}

tuner = HyperparameterTuner(
    estimator=tuner_estimator,
    objective_metric_name="validation:rmse",
    objective_type="Minimize",
    hyperparameter_ranges=hyperparameter_ranges,
    # No metric_definitions — built-in XGBoost emits validation:rmse natively.
    # The Tuner picks it up automatically.
    strategy="Bayesian",
    max_jobs=4,
    max_parallel_jobs=2,        # respects the t9 account's training-instance quota
    base_tuning_job_name=f"mlops-tune-{RUN_ID[:8]}",  # SageMaker caps tuning name at 32 chars
)

tuner.fit(
    inputs={
        "train":      TrainingInput(raw_train_uri, content_type="text/csv"),
        "validation": TrainingInput(raw_val_uri,   content_type="text/csv"),
    },
    wait=True,
    logs=False,
)
print("✓ tuning job complete:", tuner.latest_tuning_job.name)

In [ ]:
# Tuning analytics: one row per trial, columns = HP values + objective.
tuner_df = tuner.analytics().dataframe().sort_values("FinalObjectiveValue").reset_index(drop=True)
print("All trials (sorted best→worst):")
print(tuner_df[["TrainingJobName", "eta", "max_depth", "subsample",
                "FinalObjectiveValue", "TrainingJobStatus"]].to_string(index=False))

# Best trial → use these HPs in §6's pipeline training step.
best = tuner.best_training_job()
best_desc = sm_admin.describe_training_job(TrainingJobName=best)
best_hp = best_desc["HyperParameters"]
print(f"\nBest training job: {best}")
print(f"Best HPs: max_depth={best_hp['max_depth']}, eta={best_hp['eta']}, subsample={best_hp['subsample']}")

## §6. Pipelines with `ConditionStep` — branch on a metric

Real production pipelines are not just "train and register." They have a **gate**:
only register the model if it actually beats the bar. Otherwise, fail loudly so a
human investigates.

Our pipeline DAG:

```
Preprocess (Processing) ──► Train (Training) ──► Eval (Processing, emits metrics.json)
                                                      │
                                                      ▼
                                                ConditionStep
                                                rmse ≤ threshold ?
                                              yes ▼     ▼ no
                                      RegisterModel  FailStep
                                      (PendingApproval)
```

Key building blocks:

- **`PropertyFile`**: declares that the eval step writes `evaluation.json` and exposes
  its fields as referenceable properties at pipeline-build time.
- **`JsonGet`**: pulls a specific JSON path out of that property file at execution time
  so the `ConditionStep` can compare it.
- **`ConditionStep`**: branches on a `ConditionLessThanOrEqualTo` (or any of the other
  built-in conditions).
- **`FailStep`**: marks the pipeline execution as Failed with a clear error message.

In [ ]:
# Write a tiny preprocessing script the ProcessingStep will run.
# Real preprocessing would be more elaborate; for the demo we pass through with
# a small bit of feature engineering so there's something to see.
os.makedirs("scripts", exist_ok=True)

preprocess_py = '''
# Demo preprocessing: split + scale numeric columns. Output train/val/test CSVs.
import os, sys, glob
import pandas as pd
from sklearn.preprocessing import StandardScaler

in_dir  = "/opt/ml/processing/input"
out_dir = "/opt/ml/processing/output"
for split in ("train", "validation", "test"):
    os.makedirs(f"{out_dir}/{split}", exist_ok=True)

def load(name):
    path = f"{in_dir}/{name}.csv"
    return pd.read_csv(path, header=None)

train, val, test = load("train"), load("validation"), load("test")

# Column 0 is target; columns 1..N are features.
scaler = StandardScaler().fit(train.iloc[:, 1:])
for split, df in (("train", train), ("validation", val), ("test", test)):
    scaled = pd.DataFrame(scaler.transform(df.iloc[:, 1:]))
    out = pd.concat([df.iloc[:, 0].reset_index(drop=True), scaled], axis=1)
    out.to_csv(f"{out_dir}/{split}/{split}.csv", index=False, header=False)

print("✓ preprocessing done")
'''
with open("scripts/preprocessing.py", "w") as f:
    f.write(preprocess_py.strip() + "\n")

# Eval script — loads the test set + trained model, computes RMSE, writes
# /opt/ml/processing/output/evaluation.json. The condition step reads that file.
evaluate_py = '''
# Demo evaluation: compute RMSE on test set, write metrics JSON.
import json, os, tarfile, glob
import numpy as np
import pandas as pd
import xgboost as xgb

model_dir = "/opt/ml/processing/model"
test_dir  = "/opt/ml/processing/test"
out_dir   = "/opt/ml/processing/evaluation"
os.makedirs(out_dir, exist_ok=True)

# SageMaker hands us model.tar.gz — extract it.
tar_path = glob.glob(f"{model_dir}/*.tar.gz")[0]
with tarfile.open(tar_path) as t:
    t.extractall(model_dir)

# XGBoost built-in saves either xgboost-model (binary) or model.json.
candidates = glob.glob(f"{model_dir}/xgboost-model") + glob.glob(f"{model_dir}/*.json")
booster = xgb.Booster()
booster.load_model(candidates[0])

test_csv = glob.glob(f"{test_dir}/test.csv")[0]
df = pd.read_csv(test_csv, header=None)
y, X = df.iloc[:, 0].values, df.iloc[:, 1:].values
dmat = xgb.DMatrix(X)
preds = booster.predict(dmat)
rmse = float(np.sqrt(np.mean((y - preds) ** 2)))

metrics = {"regression_metrics": {"rmse": {"value": rmse}}}
with open(f"{out_dir}/evaluation.json", "w") as f:
    json.dump(metrics, f)

print(f"✓ evaluation rmse = {rmse:.4f}")
'''
with open("scripts/evaluate.py", "w") as f:
    f.write(evaluate_py.strip() + "\n")

print("✓ wrote scripts/preprocessing.py and scripts/evaluate.py")

In [ ]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterFloat
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, CacheConfig
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.properties import PropertyFile
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor

pipeline_session = PipelineSession()
cache = CacheConfig(enable_caching=True, expire_after="P30D")

# Pipeline parameter — the RMSE threshold above which we DO NOT register.
# Studio lets you override this when starting an execution.
rmse_threshold_param = ParameterFloat(name="RmseThreshold", default_value=0.7)

# ─── Preprocess step ───
pipe_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type="ml.t3.large",  # quota-safe processing instance
    instance_count=1,
    base_job_name="pipe-preprocess",
    sagemaker_session=pipeline_session,
)

step_preprocess = ProcessingStep(
    name="Preprocess",
    processor=pipe_processor,
    inputs=[ProcessingInput(source=raw_uri, destination="/opt/ml/processing/input")],
    outputs=[
        ProcessingOutput(output_name="train",      source="/opt/ml/processing/output/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/output/validation"),
        ProcessingOutput(output_name="test",       source="/opt/ml/processing/output/test"),
    ],
    code="scripts/preprocessing.py",
    cache_config=cache,
)

In [ ]:
# ─── Train step (using HPs from §5's best trial) ───
pipe_estimator = Estimator(
    image_uri=xgb_image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{PREFIX}/{RUN_ID}/pipeline-output",
    sagemaker_session=pipeline_session,
    base_job_name="pipe-train",
    max_run=900,
    hyperparameters={
        "objective":   "reg:squarederror",
        "num_round":   100,
        "max_depth":   best_hp["max_depth"],
        "eta":         best_hp["eta"],
        "subsample":   best_hp["subsample"],
        "eval_metric": "rmse",
    },
)

step_train = TrainingStep(
    name="Train",
    estimator=pipe_estimator,
    inputs={
        "train": TrainingInput(
            s3_data=step_preprocess.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv"),
        "validation": TrainingInput(
            s3_data=step_preprocess.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv"),
    },
    cache_config=cache,
)

In [ ]:
# ─── Eval step ───
# We declare a PropertyFile so the ConditionStep can JsonGet into it.
eval_property_file = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

# XGBoost-aware processor for eval. We need an image with xgboost installed —
# the simplest path is to use the XGBoost training image as a "ScriptProcessor".
from sagemaker.processing import ScriptProcessor
eval_processor = ScriptProcessor(
    image_uri=xgb_image_uri,
    role=role,
    command=["python3"],
    instance_type="ml.t3.large",
    instance_count=1,
    base_job_name="pipe-eval",
    sagemaker_session=pipeline_session,
)

step_eval = ProcessingStep(
    name="Evaluate",
    processor=eval_processor,
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_preprocess.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation"),
    ],
    code="scripts/evaluate.py",
    property_files=[eval_property_file],
    cache_config=cache,
)

In [ ]:
# ─── ConditionStep: rmse ≤ threshold ? Register : Fail ───
MODEL_PACKAGE_GROUP = "mlops-pkg-group"   # MUST match the EventBridge rule pattern in CFN

step_register = RegisterModel(
    name="RegisterModelPackage",
    estimator=pipe_estimator,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large", "ml.m5.xlarge", "ml.c5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=MODEL_PACKAGE_GROUP,
    # Pending — humans (or §7's auto-approval cell) flip this to Approved.
    approval_status="PendingManualApproval",
)

step_fail = FailStep(
    name="FailIfRmseTooHigh",
    error_message="Model RMSE above threshold — refusing to register.",
)

cond_rmse_le_threshold = ConditionLessThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=eval_property_file,
        json_path="regression_metrics.rmse.value",
    ),
    right=rmse_threshold_param,
)

step_check = ConditionStep(
    name="CheckRmse",
    conditions=[cond_rmse_le_threshold],
    if_steps=[step_register],   # rmse ≤ threshold → register
    else_steps=[step_fail],     # rmse > threshold → fail
)

# ─── Build + upsert the pipeline ───
pipeline_name = f"mlops-housing-{RUN_ID}"
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[rmse_threshold_param],
    steps=[step_preprocess, step_train, step_eval, step_check],
    sagemaker_session=pipeline_session,
)
pipeline.upsert(role_arn=role)
print("✓ pipeline upserted:", pipeline_name)

In [ ]:
# Kick off an execution. With caching enabled, a second run of the same code
# against the same inputs would mostly return CacheHit results.
execution = pipeline.start()
print("Pipeline execution ARN:", execution.arn)
execution.wait(delay=30, max_attempts=80)   # up to 40 min

print("\nStep results:")
for step in execution.list_steps():
    cache_hit = step.get("CacheHitResult", {}).get("SourcePipelineExecutionArn", "—")
    print(f"  {step['StepName']:25s}  {step['StepStatus']:10s}  cache: {cache_hit[:60]}")

## §7. Model Registry approval → EventBridge → auto-deploy

The pipeline registered a model package in `mlops-pkg-group` with status
**`PendingManualApproval`**. In a real workflow, a human reviewer (data scientist,
risk officer, ML platform team — depends on the org) inspects the metrics, the
Model Card, the lineage, and either approves or rejects.

When the package flips to **`Approved`**, the **EventBridge rule from the CFN stack**
fires and invokes the **auto-deploy Lambda**, which creates a SageMaker Model →
EndpointConfig → Endpoint. We poll until the endpoint goes `InService`, then make a
real prediction against it.

> **The pattern matters more than the demo.** This decoupling — Registry as the source
> of truth, EventBridge as the dispatcher, Lambda (or Step Functions, or CodePipeline)
> as the actuator — is how mature MLOps shops avoid "the data scientist deployed it
> to prod from their laptop at 11pm on a Friday."

In [ ]:
# Find the model package the pipeline just registered.
pkgs = sm_admin.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy="CreationTime", SortOrder="Descending", MaxResults=5,
)["ModelPackageSummaryList"]

if not pkgs:
    raise RuntimeError(
        "No model packages in mlops-pkg-group — did the pipeline's ConditionStep "
        "go down the FailStep branch? Re-run §6 with a higher RmseThreshold."
    )

latest_pkg_arn = pkgs[0]["ModelPackageArn"]
print(f"Latest package: {latest_pkg_arn}")
print(f"Current status: {pkgs[0]['ModelApprovalStatus']}")

In [ ]:
# Capture wall-clock time BEFORE approval so we can filter the endpoint
# poll below to only endpoints created after this approval — avoids picking
# up stale mlops-auto-* endpoints from previous notebook runs.
import datetime as _dt
approval_time = _dt.datetime.now(_dt.timezone.utc)
print(f"approval time : {approval_time.isoformat()}")

# Approve. This is the trigger — EventBridge picks up the state-change event,
# routes it to the Lambda based on the rule's pattern (group + status), and
# the Lambda creates Model + EndpointConfig + Endpoint.
sm_admin.update_model_package(
    ModelPackageArn=latest_pkg_arn,
    ModelApprovalStatus="Approved",
)
print("✓ status: Approved — auto-deploy Lambda should fire within seconds")

In [ ]:
# Poll for an endpoint with the mlops-auto- prefix that the Lambda creates.
# The Lambda runs in milliseconds; the Endpoint then takes ~5-7 min to
# actually come InService. CreationTimeAfter restricts to endpoints
# created after our approval — ignores any stale mlops-auto-* leftovers.
import time
auto_endpoint_name = None
for i in range(40):  # 40 * 30s = 20 min cap
    eps = sm_admin.list_endpoints(
        NameContains="mlops-auto-",
        StatusEquals="Creating",
        CreationTimeAfter=approval_time,
    )["Endpoints"] + sm_admin.list_endpoints(
        NameContains="mlops-auto-",
        StatusEquals="InService",
        CreationTimeAfter=approval_time,
    )["Endpoints"]
    if eps:
        # Pick the most recently created one (in case multiple exist).
        eps.sort(key=lambda e: e["CreationTime"], reverse=True)
        auto_endpoint_name = eps[0]["EndpointName"]
        status = eps[0]["EndpointStatus"]
        print(f"  endpoint {auto_endpoint_name} → {status} ({(i+1)*30}s elapsed)")
        if status == "InService":
            break
    else:
        print(f"  (no endpoint yet — Lambda may still be creating Model/Config) ({(i+1)*30}s elapsed)")
    time.sleep(30)

if not auto_endpoint_name:
    raise RuntimeError(
        "No mlops-auto-* endpoint appeared. Check the EventBridge rule + Lambda "
        "in the AWS console (Lambda: sagemaker-mlops-auto-deploy → Monitor → "
        "View CloudWatch logs)."
    )
print(f"✓ auto-deployed endpoint live: {auto_endpoint_name}")

In [ ]:
# Hit the live endpoint. We send a few test rows in CSV form (no header,
# no target — just features). The XGBoost container speaks text/csv natively.
sm_runtime = boto3.client("sagemaker-runtime", region_name=region)

# Pull preprocessed test rows so the input distribution matches what the
# pipeline trained on. (The pipeline scaled features in Preprocess.)
pipe_test_keys = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=f"{PREFIX}/{RUN_ID}/pipeline-output",  # estimator output_path
)

# Easier path: just send raw test rows. XGBoost is robust to small input shifts
# for a demo, and we want to prove the endpoint round-trips, not measure RMSE.
sample_rows = test_df.head(5)
payload = sample_rows.iloc[:, 1:].to_csv(index=False, header=False).strip()

resp = sm_runtime.invoke_endpoint(
    EndpointName=auto_endpoint_name,
    ContentType="text/csv",
    Accept="text/csv",
    Body=payload,
)
raw = resp["Body"].read().decode("utf-8").strip()
preds = [float(x) for x in raw.replace("\n", ",").split(",") if x]

out = sample_rows.copy()
out["pred"] = preds
out["abs_err"] = (out["MedHouseVal"] - out["pred"]).abs()
print("Live predictions from auto-deployed endpoint:")
print(out[["MedHouseVal", "pred", "abs_err"]].round(3).to_string(index=False))

## §8. Lineage Tracking + offline Feature Store query

Two things SageMaker does for free behind the scenes:

1. **Lineage**: every Processing job, Training job, Model, Model Package, Endpoint, and
   even input/output S3 artifact gets entered as a node in a graph. Edges record the
   "X used Y" relationships. We query the graph backwards from our endpoint to see
   what produced it.
2. **Offline Feature Store table**: the records ingested in §3 should now be queryable
   via Athena. Useful for: training data backfills, point-in-time joins, ad-hoc analysis.

> **In Studio:** open **Studio → Home → Lineage** (or right-click any artifact →
> "Show in lineage graph"). The same data the API surfaces below shows as a visual
> DAG with selectable nodes.

In [ ]:
# ─── Lineage: walk back from the endpoint to its training job ───
# The LineageQuery API needs a Context ARN as a starting point. SageMaker
# auto-creates a Context (type="Endpoint") for every endpoint, but the
# Context name is auto-generated, so we look it up by source_uri (the
# endpoint's own ARN). Note: EndpointContext.load() doesn't accept
# endpoint_name= directly; lookup-by-source-uri is the supported path.
from sagemaker.lineage.query import LineageQuery, LineageQueryDirectionEnum
from sagemaker.lineage.context import Context

sm_session = sagemaker.Session()

# Look up the auto-deployed endpoint's ARN from the SageMaker control plane.
ep_desc = sm_admin.describe_endpoint(EndpointName=auto_endpoint_name)
ep_arn  = ep_desc["EndpointArn"]
print(f"Endpoint ARN: {ep_arn}")

# Find the Lineage Context whose source is this endpoint.
contexts = list(Context.list(source_uri=ep_arn, sagemaker_session=sm_session))
ep_ctx = next((c for c in contexts if c.context_type == "Endpoint"), None)
if ep_ctx is None:
    raise RuntimeError(f"No Lineage Context found for endpoint {auto_endpoint_name}")
print(f"Endpoint Lineage Context: {ep_ctx.context_name}")

# Walk all ascendants (everything upstream of this endpoint). The QueryLineage
# API requires either include_edges=True OR a LineageFilter — we pass True
# since edges are cheap and we'll just iterate vertices below.
upstream = LineageQuery(sm_session).query(
    start_arns=[ep_ctx.context_arn],
    direction=LineageQueryDirectionEnum.ASCENDANTS,
    include_edges=True,
    max_depth=10,
)

print(f"\nAscendants of {auto_endpoint_name}: {len(upstream.vertices)} vertices, {len(upstream.edges)} edges")
seen_types = {}
for v in upstream.vertices:
    # Vertex exposes .lineage_entity (e.g. Artifact, Action, Context, TrialComponent)
    # and .lineage_source (e.g. Image, ModelArtifact, TrainingJob).
    seen_types.setdefault(v.lineage_entity, []).append((v.lineage_source, v.arn))
for entity_type, items in sorted(seen_types.items()):
    print(f"\n  {entity_type}  ({len(items)} entities)")
    for source_type, arn in items[:5]:
        print(f"    [{source_type}]  {arn[-90:]}")

In [ ]:
# ─── Offline Feature Store query via Athena ───
# FeatureGroup.athena_query() returns a helper bound to the right table + database.
fs_query = feature_group.athena_query()
print(f"Athena database: {fs_query.database}")
print(f"Athena table   : {fs_query.table_name}")

# Feature Store needs at least 5-15 min between ingest and offline visibility.
# By now (~30+ min after §3) the data should have flushed.
sql = f'''
SELECT
  COUNT(*)         AS row_count,
  AVG("medhouseval") AS avg_target,
  MIN("medhouseval") AS min_target,
  MAX("medhouseval") AS max_target
FROM "{fs_query.table_name}"
'''

# Athena needs a query results location. We'll use the offline FS bucket.
athena_results = f"s3://{bucket}/{PREFIX}/{RUN_ID}/athena-results/"

fs_query.run(query_string=sql, output_location=athena_results)
fs_query.wait()
result_df = fs_query.as_dataframe()
print("\nOffline FS aggregate query result:")
print(result_df.to_string(index=False))

## §9. Class discussion prompts

1. **Bayesian vs random tuning budget:** we ran 4 trials, max 2 in parallel. At what
   trial count does Bayesian really start to beat random? What's the break-even point
   in your team's compute economics?

2. **Feature Store online vs offline cost:** online store is DynamoDB — milliseconds
   but priced per RCU/WCU. Offline is S3 + Athena — minutes, but pennies. How would
   you decide which features go online?

3. **ConditionStep `else_steps`:** we chose `FailStep`. What other branches make
   sense in production? (Hint: register-with-different-status, retrain-with-more-data,
   page-the-on-call, alternate model fallback.)

4. **Approval workflow access control:** who in your org should be allowed to flip
   `ModelApprovalStatus` to `Approved`? What gates do you want in front of that —
   SCP? Tag-based IAM condition? Dual-control (two-person approval)?

5. **Lineage as audit trail:** if regulators asked "what data trained the model that
   made this decision?", how would you answer in your current setup? How does Lineage
   Tracking change that answer?

## §10. Cleanup — **don't skip this**

The auto-deployed endpoint is the costly bit (~$0.10/h on `ml.m5.large`). Everything
else is pennies. Run all cells below before you log off. The CFN stack itself
(Lambda, EventBridge rule, Studio domain, S3 bucket) is left in place — tear it
down with `cleanup_pre_demo.sh` + `aws cloudformation delete-stack` from your
laptop.

In [ ]:
# 10.1 — delete the auto-deployed endpoint + endpoint config + model
if auto_endpoint_name:
    try:
        ep_desc = sm_admin.describe_endpoint(EndpointName=auto_endpoint_name)
        ep_cfg  = ep_desc["EndpointConfigName"]
        cfg_desc = sm_admin.describe_endpoint_config(EndpointConfigName=ep_cfg)
        model_name = cfg_desc["ProductionVariants"][0]["ModelName"]

        sm_admin.delete_endpoint(EndpointName=auto_endpoint_name)
        print(f"deleted endpoint {auto_endpoint_name}")
        sm_admin.delete_endpoint_config(EndpointConfigName=ep_cfg)
        print(f"deleted endpoint-config {ep_cfg}")
        sm_admin.delete_model(ModelName=model_name)
        print(f"deleted model {model_name}")
    except Exception as e:
        print(f"endpoint cleanup: {e}")

In [ ]:
# 10.2 — delete the model package(s) + group
# Order matters: packages must go before the group.
try:
    for p in sm_admin.list_model_packages(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    )["ModelPackageSummaryList"]:
        sm_admin.delete_model_package(ModelPackageName=p["ModelPackageArn"])
        print(f"deleted model-package {p['ModelPackageArn'][-60:]}")
    sm_admin.delete_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP)
    print(f"deleted model-package-group {MODEL_PACKAGE_GROUP}")
except Exception as e:
    print(f"package cleanup: {e}")

In [ ]:
# 10.3 — delete the pipeline
try:
    sm_admin.delete_pipeline(PipelineName=pipeline_name)
    print(f"deleted pipeline {pipeline_name}")
except Exception as e:
    print(f"pipeline cleanup: {e}")

In [ ]:
# 10.4 — delete the FeatureGroup (drops the online DDB table; offline S3 stays
# so historical Athena queries keep working — sweep it via the cleanup script).
try:
    feature_group.delete()
    print(f"deleted feature-group {fg_name}")
except Exception as e:
    print(f"feature-group cleanup: {e}")

In [ ]:
# 10.5 — sweep the run-scoped S3 prefix.
s3_resource = boto3.resource("s3", region_name=region)
bucket_obj  = s3_resource.Bucket(bucket)
deleted = 0
for obj in bucket_obj.objects.filter(Prefix=f"{PREFIX}/{RUN_ID}/"):
    obj.delete(); deleted += 1
print(f"deleted {deleted} S3 objects under {PREFIX}/{RUN_ID}/")

---

### Recap

In ~90 minutes you exercised every MLOps surface SageMaker exposes:

- **Feature Store** — online (DDB, sub-100ms) + offline (S3 + Athena) from one ingest call.
- **Experiments** — three tracked training runs with auto-logged params & metrics.
- **Hyperparameter Tuning** — Bayesian search over a 3-D parameter space.
- **Pipelines + ConditionStep** — `Preprocess → Train → Eval → branch` with FailStep.
- **Model Registry approval workflow** — Pending → Approved → **EventBridge → Lambda → live endpoint**.
- **Lineage Tracking** — endpoint walked back through model package → training job → input artifacts.

Map back to MLOps lifecycle: **data prep** (Feature Store), **experimentation**
(Experiments + HPO), **build** (Pipelines), **release** (Registry + EventBridge),
**operate** (live endpoint), **govern** (Lineage). The last piece — **monitor** — is
covered in the comprehensive demo's §10 (`DefaultModelMonitor` + `DataCaptureConfig`).

Verify everything is gone:
```
aws --profile t9 sagemaker list-endpoints --query 'Endpoints[?starts_with(EndpointName,`mlops-auto-`)]'
aws --profile t9 sagemaker list-feature-groups --query 'FeatureGroupSummaries[?contains(FeatureGroupName,`mlops-housing`)]'
aws --profile t9 sagemaker list-pipelines --query 'PipelineSummaries[?contains(PipelineName,`mlops-housing`)]'
```
All three should return `[]`. The CFN-managed Lambda + EventBridge rule are left
intact — tear them down via `cleanup_pre_demo.sh` + `cfn delete-stack` from your laptop.